# Cortex-M backend flow example - Running MobileNetV3 in float16

This notebook demonstrates an end-to-end **ExecuTorch + Cortex-M float16** flow on **Corstone-300** using **MobileNetV3-Small**.

It covers:
- choosing and visualizing a random Imagenette sample
- running host-side PyTorch inference and showing the top-10 probabilities
- building the Cortex-M backend with CMSIS-NN float16 enabled
- exporting a `.pte` and semihosting assets
- building the semihosting runner
- launching FVP directly from the notebook
- comparing the top-10 PyTorch and FVP probabilities and plotting their difference

## Before you begin

1. **Run Jupyter from the local ExecuTorch virtual environment.**
   Example:
   ```bash
   cd /path/to/executorch
   source venv/bin/activate
   jupyter notebook examples/arm/cortex_m_mv3_f16_example.ipynb
   ```
2. Install the extra notebook dependencies into that same local virtual environment.
   Example:
   ```bash
   pip install notebook ipykernel matplotlib datasets pyyaml
   ```
3. Install Arm toolchains and FVP once using the usual setup flow.
4. Make sure you have a **local CMSIS-NN checkout with float support**.
   At the time of writing, the float path is not available from the default upstream `main` checkout used by many builds.
5. Update the configuration cells below before running the notebook.

## Notes

- The Cortex-M float backend is still more experimental than the quantized path.
- This notebook defaults to **float16** and uses a **channels-last** input tensor.
- Recommended toolchains for this path are recent **Arm Compiler 6** or **Arm Toolchain for Embedded (LLVM)**.


In [ ]:
from collections import Counter
from pathlib import Path
import json
import importlib
import os
import random
import re
import shlex
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from datasets import load_dataset

# Load the mobilenet_v3_small model and its weights
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small


### ExecuTorch Cortex-M backend CMSIS-NN operators

In [ ]:

from executorch.backends.cortex_m.passes.cortex_m_pass_manager import CortexMPassManager
from executorch.exir import EdgeCompileConfig
from executorch.extension.export_util.utils import export_to_edge, save_pte_program


## Repository Root


In [ ]:
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'backends').exists() and (candidate / 'examples').exists():
        REPO_ROOT = candidate
        break
else:
    raise AssertionError(f'Could not locate ExecuTorch repo root from: {cwd}')

print(f'Resolved repo root: {REPO_ROOT}')


## Toolchain Configuration


In [ ]:
TOOLCHAIN = 'arm-none-eabi-gcc'  # one of: arm-none-eabi-gcc, armclang, clang
TOOLCHAIN_BIN = Path('/path/to/toolchain/bin').expanduser()
assert TOOLCHAIN_BIN.exists(), f'Missing toolchain bin dir: {TOOLCHAIN_BIN}'

FVP_BIN = shutil.which('FVP_Corstone_SSE-300_Ethos-U55')
assert FVP_BIN is not None, 'Could not find FVP_Corstone_SSE-300_Ethos-U55 in PATH'
FVP_BIN = Path(FVP_BIN).resolve()
print(f'Using FVP executable: {FVP_BIN}')

## CMSIS-NN Checkout path
- Temp : CMSIS-NN float support is not yet part of main branch


In [ ]:
CMSIS_NN_LOCAL_PATH = Path('/path/to/CMSIS-NN').expanduser().resolve()
assert CMSIS_NN_LOCAL_PATH.exists(), f'Missing CMSIS-NN checkout: {CMSIS_NN_LOCAL_PATH}'


## Run Configuration


In [ ]:
ET_BUILD_ROOT = REPO_ROOT / 'arm_test_mv3_f16_notebook'
RUN_DIR = ET_BUILD_ROOT / 'mobilenet_v3_small_float16_semihosting_run'
RUNNER_DIR = ET_BUILD_ROOT / 'mobilenet_v3_small_semihosting_runner'
TIMEOUT_SECONDS = 1800  # FVP timeout in seconds


### Imagenet index sample (None = random)

In [ ]:
SAMPLE_SEED = 11
SAMPLE_INDEX = 10

## Helper Utilities


- set virtual env path

In [ ]:
VENV_PYTHON = REPO_ROOT / 'venv/bin/python'

- track build log output to notebook, top-10 output probabilities

In [ ]:

assert VENV_PYTHON.exists(), f'Missing venv python: {VENV_PYTHON}'

WEIGHTS = MobileNet_V3_Small_Weights.DEFAULT
CATEGORIES = WEIGHTS.meta['categories']
PREPROCESS = WEIGHTS.transforms()


def command_env(extra_env: dict | None = None) -> dict[str, str]:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_ROOT / 'src')
    env['PATH'] = f"{TOOLCHAIN_BIN}:{env.get('PATH', '')}"
    if TOOLCHAIN == 'armclang':
        env['AC6_TOOLCHAIN'] = str(TOOLCHAIN_BIN)
    elif TOOLCHAIN == 'clang':
        env['CLANG_TOOLCHAIN_ROOT'] = str(TOOLCHAIN_BIN)
    if extra_env:
        env.update(extra_env)
    return env


def run_cmd(args: list[str], extra_env: dict | None = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(
        args,
        cwd=REPO_ROOT,
        env=command_env(extra_env),
        check=check,
        text=True,
        capture_output=True,
    )


def resolve_runner_executable(runner_dir: Path) -> Path:
    for name in ('arm_executor_runner', 'arm_executor_runner.elf'):
        candidate = runner_dir / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Runner executable not found in {runner_dir}')


def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)


def topk_probs(logits: np.ndarray, k: int = 10):
    probs = softmax_np(logits)
    top_idx = np.argsort(-probs)[:k]
    return top_idx, probs[top_idx], probs


RUN_DIR.mkdir(parents=True, exist_ok=True)
RUNNER_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo root      : {REPO_ROOT}')
print(f'Toolchain      : {TOOLCHAIN}')
print(f'Build root     : {ET_BUILD_ROOT}')
print(f'Run dir        : {RUN_DIR}')
print(f'CMSIS-NN local : {CMSIS_NN_LOCAL_PATH}')


def run_stream(
    args: list[str],
    extra_env: dict | None = None,
    heartbeat_s: int = 30,
    log_file: Path | None = None,
    echo_filter: tuple[str, ...] | None = ('Built target', 'Generating', 'Installing:', '[', '%]'),
) -> str:
    print('Starting command:')
    print(shlex.join(args))
    proc = subprocess.Popen(
        args,
        cwd=REPO_ROOT,
        env=command_env(extra_env),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert proc.stdout is not None
    last_update = time.monotonic()
    captured: list[str] = []
    log_handle = None
    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        log_handle = log_file.open('w', encoding='utf-8')

    try:
        while True:
            line = proc.stdout.readline()
            if line:
                captured.append(line)
                if log_handle is not None:
                    log_handle.write(line)
                    log_handle.flush()
                line_stripped = line.rstrip()
                if line_stripped:
                    if echo_filter is None:
                        print(line_stripped)
                    elif any(token in line_stripped for token in echo_filter):
                        print(line_stripped)
                last_update = time.monotonic()
                continue

            if proc.poll() is not None:
                break

            now = time.monotonic()
            if now - last_update >= heartbeat_s:
                print('... command still running ...')
                last_update = now
            time.sleep(1.0)
    finally:
        if log_handle is not None:
            log_handle.close()

    ret = proc.wait()
    output = ''.join(captured)
    if ret != 0:
        raise subprocess.CalledProcessError(ret, args, output=output)
    print('Command completed successfully.')
    return output


## Load and visualize a sample

The Cortex-M float flow uses a rank-4 **channels-last** tensor. We first pick a random Imagenette validation image, display it, then apply the MobileNetV3 preprocessing transforms.


## Choose the Sample Index


In [ ]:
dataset = load_dataset('frgfm/imagenette', 'full_size', split='validation')
if SAMPLE_INDEX is None:
    CHOSEN_INDEX = random.Random(SAMPLE_SEED).randrange(len(dataset))
else:
    CHOSEN_INDEX = int(SAMPLE_INDEX)
print(f'Chosen sample index: {CHOSEN_INDEX}')


In [ ]:
sample = dataset[CHOSEN_INDEX]
label_names = dataset.features['label'].names
true_label = label_names[sample['label']]

original_img = sample['image'].convert('RGB')
transformed = PREPROCESS(original_img).unsqueeze(0).to(memory_format=torch.channels_last)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(original_img)
axes[0].set_title(f'Original image\nlabel={true_label}')
axes[0].axis('off')

plottable = transformed.squeeze(0).permute(1, 2, 0)
plottable = (plottable - plottable.min()) / (plottable.max() - plottable.min())
axes[1].imshow(plottable)
axes[1].set_title('Transformed input')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'True label   : {true_label}')
print(f'Input shape  : {tuple(transformed.shape)}')
print(f'Channels last: {transformed.is_contiguous(memory_format=torch.channels_last)}')


## PyTorch MobileNetV3 inference

Run the host-side float16 model first and keep the logits for later comparison.


In [ ]:
model = mobilenet_v3_small(weights=WEIGHTS).eval().half()
sample_tensor = transformed.to(torch.float16)
with torch.no_grad():
    pytorch_output = model(sample_tensor)

pytorch_logits = pytorch_output[0].detach().cpu().to(torch.float32).numpy()
pt_top_idx, pt_top_vals, pytorch_probs = topk_probs(pytorch_logits, k=10)

print(f'PyTorch top-1: {CATEGORIES[int(pytorch_logits.argmax())]}')
for rank, (idx, value) in enumerate(zip(pt_top_idx.tolist(), pt_top_vals.tolist()), start=1):
    print(f'{rank:2d}. {CATEGORIES[idx]}: {value:.6f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(pt_top_idx)), pt_top_vals)
ax.set_xticks(range(len(pt_top_idx)), [CATEGORIES[i] for i in pt_top_idx], rotation=45, ha='right')
ax.set_ylabel('Probability')
ax.set_title('PyTorch top-10 probabilities')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Build the Cortex-M backend with float16 enabled

This cell runs the same backend build step as the semihosting wrapper, but explicitly and only for `float16`.


In [ ]:
build_args = [
    str(REPO_ROOT / 'backends/arm/scripts/build_executorch.sh'),
    f'--toolchain={TOOLCHAIN}',
    f'--et_build_root={ET_BUILD_ROOT}',
    f'--cmsis_nn_local_path={CMSIS_NN_LOCAL_PATH}',
    '--cmsis_nn_enable_f16',
    '--devtools',
]
run_stream(build_args)


## Inspect the generated float capability JSON

The build writes a small JSON artifact into the build tree. Python lowering reads this file so it knows exactly which Cortex-M float dtypes were compiled into the backend.


In [ ]:
build_dir = ET_BUILD_ROOT / {
    'arm-none-eabi-gcc': 'cmake-out',
    'armclang': 'cmake-out-armclang',
    'clang': 'cmake-out-clang',
}[TOOLCHAIN]
capabilities_json = build_dir / 'backends/cortex_m/float_capabilities.json'
print(capabilities_json)
print(capabilities_json.read_text())


## Export the `.pte` and semihosting assets

This notebook now runs the MobileNetV3 export/lowering step in a fresh Python subprocess.
That keeps the notebook robust even if the current Jupyter kernel already imported ExecuTorch
modules with operator registrations.

For the reusable scripted version, see:
- `backends/cortex_m/test/models/export_float_mobilenet_v3_demo.py`

The build already generated `float_capabilities.json`; this export step passes that file into the
subprocess so the Cortex-M lowering pipeline only emits float ops that were actually compiled.


In [ ]:
os.environ['EXECUTORCH_CORTEX_M_FLOAT_CAPABILITIES_FILE'] = str(capabilities_json)
print('Using float capability file:', os.environ['EXECUTORCH_CORTEX_M_FLOAT_CAPABILITIES_FILE'])

export_snippet = f"""
import json
import os
from pathlib import Path

import torch
from datasets import load_dataset
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

import executorch.backends.cortex_m.ops.operators  # noqa: F401
from executorch.backends.cortex_m.passes.cortex_m_pass_manager import CortexMPassManager
from executorch.exir import EdgeCompileConfig
from executorch.extension.export_util.utils import export_to_edge, save_pte_program


def write_storage_bytes(path: Path, tensor: torch.Tensor) -> None:
    path.write_bytes(bytes(tensor.untyped_storage()))

run_dir = Path({str(RUN_DIR)!r})
run_dir.mkdir(parents=True, exist_ok=True)
weights = MobileNet_V3_Small_Weights.DEFAULT
preprocess = weights.transforms()
dataset = load_dataset('frgfm/imagenette', 'full_size', split='validation')
sample = dataset[{int(CHOSEN_INDEX)}]
label_names = dataset.features['label'].names
true_label = label_names[sample['label']]
image = sample['image'].convert('RGB')
sample_tensor = preprocess(image).unsqueeze(0).to(memory_format=torch.channels_last).to(torch.float16)

model = mobilenet_v3_small(weights=weights).eval().half()
with torch.no_grad():
    pytorch_output = model(sample_tensor)

edge_program = export_to_edge(
    model,
    (sample_tensor,),
    edge_compile_config=EdgeCompileConfig(_check_ir_validity=False),
)
transformed_program = CortexMPassManager(edge_program.exported_program()).transform()
edge_program._edge_programs['forward'] = transformed_program
program = edge_program.to_executorch()

save_pte_program(program, 'mobilenet_v3_small_f16_demo', str(run_dir))
write_storage_bytes(run_dir / 'i0.bin', sample_tensor)
write_storage_bytes(run_dir / 'expected-0.bin', pytorch_output)

meta = {{
    'dtype': 'float16',
    'true_label': true_label,
    'predicted_label': weights.meta['categories'][int(pytorch_output.argmax().item())],
    'sample_index': {int(CHOSEN_INDEX)},
}}
(run_dir / 'run_meta.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

# Also print a small operator summary for notebook visibility.
graph_module = transformed_program.graph_module
counts = {{}}
for node in graph_module.graph.nodes:
    target = getattr(node, 'target', None)
    if target is None:
        continue
    name = getattr(target, '__name__', str(target))
    counts[name] = counts.get(name, 0) + 1

print('Lowered Cortex-M operators:')
for name, count in sorted((k, v) for k, v in counts.items() if 'cortex_m' in k):
    print(f'  {{count:2d}} x {{name}}')
print('Remaining aten operators:')
aten_items = [(k, v) for k, v in counts.items() if k.startswith('executorch_exir_dialects_edge__ops_aten_')]
if aten_items:
    for name, count in sorted(aten_items):
        print(f'  {{count:2d}} x {{name}}')
else:
    print('  none')
print('Remaining layout helpers:')
layout_items = [(k, v) for k, v in counts.items() if 'dim_order' in k or 'view_copy' in k]
if layout_items:
    for name, count in sorted(layout_items):
        print(f'  {{count:2d}} x {{name}}')
else:
    print('  none')
"""

res = run_cmd(
    [str(VENV_PYTHON), '-c', export_snippet],
    extra_env={'EXECUTORCH_CORTEX_M_FLOAT_CAPABILITIES_FILE': str(capabilities_json)},
)
print(res.stdout)

pte_file = RUN_DIR / 'mobilenet_v3_small_f16_demo.pte'
meta = json.loads((RUN_DIR / 'run_meta.json').read_text())
print(f'PTE file: {pte_file}')
print(json.dumps(meta, indent=2))


## Generate the selected-op list and build the semihosting runner

This mirrors the wrapper flow:
1. generate `selected_ops.yaml`
2. extract the portable `aten::` / `dim_order_ops::` fallback ops
3. build the semihosting runner


In [ ]:
gen_oplist_args = [
    str(VENV_PYTHON),
    str(REPO_ROOT / 'codegen/tools/gen_oplist.py'),
    f'--model_file_path={pte_file}',
    f'--output_path={RUN_DIR / "selected_ops.yaml"}',
]
res = run_cmd(gen_oplist_args)
print(res.stdout or 'selected_ops.yaml generated')

data = yaml.safe_load((RUN_DIR / 'selected_ops.yaml').read_text())
select_ops = sorted(
    op for op in data.get('operators', {}).keys()
    if op.startswith('aten::') or op.startswith('dim_order_ops::')
)
select_ops_list = ','.join(select_ops)
print('Selected portable ops:')
for op in select_ops:
    print(' ', op)

runner_args = [
    str(REPO_ROOT / 'backends/arm/scripts/build_executor_runner.sh'),
    f'--toolchain={TOOLCHAIN}',
    f'--et_build_root={ET_BUILD_ROOT}',
    '--pte=semihosting',
    '--target=ethos-u55-128',
    '--system_config=Ethos_U55_High_End_Embedded',
    '--memory_mode=Shared_Sram',
    f'--output={RUNNER_DIR}',
    f'--select_ops_list={select_ops_list}',
    '--extra_build_flags=-DET_ARM_BAREMETAL_METHOD_ALLOCATOR_POOL_SIZE=83886080',
]
run_stream(runner_args)

runner_exe = resolve_runner_executable(RUNNER_DIR)
print(f'Runner: {runner_exe}')


## Construct the direct FVP command

This is the exact style of command used by the semihosting wrappers. The next cell runs it directly from the notebook.


In [ ]:
fvp_args = [
    str(FVP_BIN),
    '-C', 'ethosu.num_macs=128',
    '-C', 'mps3_board.visualisation.disable-visualisation=1',
    '-C', 'mps3_board.telnetterminal0.start_telnet=0',
    '-C', "mps3_board.uart0.out_file=-",
    '-C', 'cpu0.semihosting-enable=1',
    '-C', 'cpu0.semihosting-stack_base=0',
    '-C', 'cpu0.semihosting-heap_limit=0',
    '-C', f'cpu0.semihosting-cwd={RUN_DIR}',
    '-C', 'ethosu.extra_args=--fast',
    '-C', f'cpu0.semihosting-cmd_line=executor_runner -m {pte_file.name} -o out -i i0.bin',
    '-a', str(runner_exe),
    '--timelimit', str(TIMEOUT_SECONDS),
]
print(shlex.join(fvp_args))


## Run Corstone-300 FVP

This cell streams the FVP UART output live in the notebook and also saves the full log to `run.log` in the run directory.


In [ ]:
run_log = RUN_DIR / 'run.log'
res_stdout = run_stream(fvp_args, log_file=run_log, echo_filter=None)
print(res_stdout[-4000:])
print(f'Full log saved to: {run_log}')

cycle_match = re.search(r'Inference runtime: ([0-9]+) CPU cycles', res_stdout)
if cycle_match:
    print(f'Inference cycles: {cycle_match.group(1)}')
else:
    print('Could not find cycle count in FVP output.')


## Compare PyTorch and FVP outputs

The FVP run writes `out-0.bin`. The helper already wrote the reference output in `expected-0.bin`.

The plots below show:
- PyTorch top-10 probabilities
- FVP top-10 probabilities for the same classes
- absolute difference on the same top-10 set


In [ ]:
expected = np.fromfile(RUN_DIR / 'expected-0.bin', dtype=np.float16).astype(np.float32)
got = np.fromfile(RUN_DIR / 'out-0.bin', dtype=np.float16).astype(np.float32)

assert expected.shape == got.shape

ref_probs = softmax_np(expected)
got_probs = softmax_np(got)
common_top_idx = np.argsort(-ref_probs)[:10]

print(f'PyTorch top-1: {CATEGORIES[int(expected.argmax())]}')
print(f'FVP top-1    : {CATEGORIES[int(got.argmax())]}')
print(f'max_abs      : {np.max(np.abs(got - expected))}')
print(f'mean_abs     : {np.mean(np.abs(got - expected))}')

labels = [CATEGORIES[i] for i in common_top_idx]
ref_vals = ref_probs[common_top_idx]
got_vals = got_probs[common_top_idx]
diff_vals = np.abs(got_vals - ref_vals)

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)

axes[0].bar(range(10), ref_vals)
axes[0].set_title('PyTorch top-10')
axes[0].set_ylabel('Probability')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(range(10), got_vals)
axes[1].set_title('FVP top-10')
axes[1].set_ylabel('Probability')
axes[1].grid(True, axis='y', alpha=0.3)

axes[2].bar(range(10), diff_vals)
axes[2].set_title('|FVP - PyTorch|')
axes[2].set_ylabel('Probability diff')
axes[2].set_xticks(range(10), labels, rotation=45, ha='right')
axes[2].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## Summary

If the notebook ran cleanly, you have now reproduced the full MobileNetV3 float16 Cortex-M semihosting flow directly from Jupyter:

- random Imagenette sample selection and visualization
- host-side PyTorch inference and top-10 probabilities
- Cortex-M backend build with CMSIS-NN float16 enabled
- `.pte` export and semihosting asset generation
- semihosting runner build
- direct Corstone-300 FVP execution
- PyTorch vs FVP output comparison

This notebook is intentionally close to the existing semihosting wrapper flow so the steps can be debugged one by one.
